# TasvirEt Encoder Deneyleri (E1–E4)

| ID | Encoder | Pretrained ağırlık |
|---|---|---|
| E1 | MobileCLIP-S0 | `checkpoints/mobileclip_s0.pt` |
| E2 | MobileCLIP-S1 | `checkpoints/mobileclip_s1.pt` |
| E3 | MobileCLIP2-S0 | `checkpoints/mobileclip2_s0.pt` |
| E4 | MobileCLIP2-S2 | `checkpoints/mobileclip2_s2.pt` |

Her deney bağımsızdır: kendi pretrained encoder'ı + `dbmdz/bert-base-turkish-cased` ile başlar,
Stage 1 (250 it, sadece MLP2) → Stage 2 (16.000 it, MLP2 + decoder; encoder her zaman frozen).
Checkpoint'ler: 4k / 8k / 12k / 16k (yalnızca model ağırlığı). Best = val CIDEr.
Test split: best-val ve 16k checkpoint'i. Tüm ayarlar: `configs/encoder_exp/encoder_experiments.yaml`.

## 1. Ortam (bir kez)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf /content/TRCAP
!git clone -b encoder-exp https://github.com/mhr871/TRCAP.git
%cd /content/TRCAP
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -r requirements_colab.txt
!python -m pip install --no-deps git+https://github.com/apple/ml-mobileclip.git
!python -c "import torch, transformers, timm, open_clip, mobileclip; print('torch', torch.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__, '| open_clip', open_clip.__version__, '| cuda', torch.cuda.is_available())"

## 2. Veri ve pretrained ağırlıklar (bir kez)

In [ ]:
!python tools/prepare_tasviret.py --allow-missing-images
!python tools/download_tasviret_images.py
!python tools/prepare_tasviret.py --images-root Data/flickr8k/images

In [ ]:
!python tools/download_mobileclip.py --model mobileclip_s0 mobileclip_s1 mobileclip2_s0 mobileclip2_s2

## 3. Preflight (eğitim başlatmaz)
Dört encoder için ağırlık yükleme, stage 1/stage 2 forward+backward, frozen kontrolleri ve tam batch VRAM testi.
`PREFLIGHT PASSED` görmeden 4. adıma geçmeyin.

In [ ]:
!PYTHONPATH=/content/TRCAP python run_encoder_experiments.py --preflight

## 4. E1 → E2 → E3 → E4 eğitimi (tek hücre, sırayla)
Çıktılar: `/content/drive/MyDrive/TRCAP_encoder_exp/<ID>_<Encoder>/{checkpoints,logs,metrics,config}`.
Ortak sonuç dosyaları: `encoder_results.csv` / `encoder_results.json`. Her deney bitince hemen yazılır.
Klasörde checkpoint varsa eğitim başlamaz (bilerek silip yeniden koşmak için `--overwrite`).

In [ ]:
!PYTHONPATH=/content/TRCAP python -u run_encoder_experiments.py   --save-dir /content/drive/MyDrive/TRCAP_encoder_exp

## 5. Sonuç tablosu

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TRCAP_encoder_exp/encoder_results.csv')
cols = ['experiment_id', 'encoder', 'status', 'encoder_params', 'best_iter', 'best_val_CIDEr',
        'test_best_CIDEr', 'test_best_Bleu_4', 'test_best_ROUGE_L',
        'test_final_CIDEr', 'test_final_Bleu_4', 'test_final_ROUGE_L', 'training_time']
df[cols]